### Load every season into one table

Read each Premier League CSV in `data/raw` (files named `E*.csv`), drop blank rows, tag every row with its season code, and stack the seasons into a single `matches` table.

In [2]:
from pathlib import Path
import pandas as pd

raw_dir = Path("/Users/mac/football-prediction/data/raw")
frames = []

for file in sorted(raw_dir.glob("E*.csv")):
    season = file.stem[1:]                    # "E1718" -> "1718"
    df = pd.read_csv(file, encoding="latin-1")
    df = df.dropna(subset=["HomeTeam"])       # remove blank rows
    df["Season"] = season
    frames.append(df)

matches = pd.concat(frames, ignore_index=True)

### Table size

Show how many matches (rows) and columns are in the combined table.

In [3]:
matches.shape

(3520, 196)

### Matches per season

Count how many matches fall in each season.

In [4]:
matches['Season'].value_counts()

Season
1718    380
1819    380
1920    380
2021    380
2122    380
2223    380
2324    380
2425    380
2526    380
2627    100
Name: count, dtype: int64

### Keep the columns we need

Keep season, date, teams, full-time score and result, shots, fouls, corners, cards, and Bet365 home/draw/away odds. Drop every other column.

In [5]:
core_cols = [
    "Season", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HS", "AS", "HST", "AST",
    "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR",
    "B365H", "B365D", "B365A",
]
matches = matches[core_cols]

### Preview the filtered table

Display the matches table after the extra columns were removed.

In [6]:
matches

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,...,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A
0,1718,11/08/2017,Arsenal,Leicester,4,3,H,27,6,10,...,12,9,4,0,1,0,0,1.53,4.50,6.50
1,1718,12/08/2017,Brighton,Man City,0,2,A,6,14,2,...,9,3,10,0,2,0,0,11.00,5.50,1.33
2,1718,12/08/2017,Chelsea,Burnley,2,3,A,19,10,6,...,11,8,5,3,3,2,0,1.25,6.50,15.00
3,1718,12/08/2017,Crystal Palace,Huddersfield,0,3,A,14,8,4,...,19,12,9,1,3,0,0,1.83,3.60,5.00
4,1718,12/08/2017,Everton,Stoke,1,0,H,9,9,4,...,10,6,7,1,1,0,0,1.70,3.80,5.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3515,2627,08/11/2026,Brighton,Brentford,2,0,H,20,5,6,...,7,7,3,4,2,0,1,1.90,3.80,3.70
3516,2627,08/11/2026,Crystal Palace,Liverpool,1,3,A,19,20,7,...,18,4,7,2,4,0,0,5.50,4.33,1.57
3517,2627,08/11/2026,Ipswich,Bournemouth,1,2,A,10,15,2,...,20,2,7,2,3,0,0,3.80,3.75,1.91
3518,2627,08/11/2026,Sunderland,Chelsea,1,0,H,9,10,2,...,8,2,2,5,4,0,0,3.90,3.60,1.91


### Look at season 2627

Show only the rows whose season code is `2627`.

In [7]:
matches[matches['Season']== '2627']

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,...,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A
3420,2627,21/08/2026,Arsenal,Coventry,3,0,H,20,4,6,...,13,8,2,1,1,0,0,1.20,7.00,13.00
3421,2627,22/08/2026,Hull,Man United,2,0,H,8,21,4,...,9,1,6,2,1,0,0,8.50,5.00,1.36
3422,2627,22/08/2026,Everton,Crystal Palace,2,0,H,16,11,5,...,15,4,0,0,1,0,0,2.15,3.50,3.30
3423,2627,22/08/2026,Ipswich,Sunderland,2,1,H,10,11,3,...,14,3,5,0,3,0,0,2.70,3.30,2.63
3424,2627,22/08/2026,Nott'm Forest,Leeds,0,1,A,12,11,2,...,14,3,2,2,1,0,0,2.30,3.40,3.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3515,2627,08/11/2026,Brighton,Brentford,2,0,H,20,5,6,...,7,7,3,4,2,0,1,1.90,3.80,3.70
3516,2627,08/11/2026,Crystal Palace,Liverpool,1,3,A,19,20,7,...,18,4,7,2,4,0,0,5.50,4.33,1.57
3517,2627,08/11/2026,Ipswich,Bournemouth,1,2,A,10,15,2,...,20,2,7,2,3,0,0,3.80,3.75,1.91
3518,2627,08/11/2026,Sunderland,Chelsea,1,0,H,9,10,2,...,8,2,2,5,4,0,0,3.90,3.60,1.91


### Parse match dates

Turn the `Date` text into real datetimes. Dates are read day-first, and mixed formats in the files are accepted.

In [8]:
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True, format="mixed")

### Check the dates parsed

Confirm `Date` is a datetime column and that no date failed to parse (the missing count should be 0).

In [9]:
matches["Date"].dtype          # should be datetime64[ns]
matches["Date"].isna().sum()   # should be 0

np.int64(0)

### Date range of each season

For every season, show the earliest and latest match date.

In [10]:
matches.groupby("Season")["Date"].agg(["min", "max"])

,min,max
Season,,
1718,2017-08-11,2018-05-13
1819,2018-08-10,2019-05-12
1920,2019-08-09,2020-07-26
2021,2020-09-12,2021-05-23
2122,2021-08-13,2022-05-22
2223,2022-08-05,2023-05-28
2324,2023-08-11,2024-05-19
2425,2024-08-16,2025-05-25
2526,2025-08-15,2026-05-24


### Date range of the whole dataset

Show the earliest and latest match date across all seasons.

In [11]:
matches["Date"].min(), matches["Date"].max()

(Timestamp('2017-08-11 00:00:00'), Timestamp('2026-11-08 00:00:00'))

### Sort matches by date

Order the table from oldest match to newest and reset the row numbers.

In [12]:
matches = matches.sort_values("Date").reset_index(drop=True)

### Look for duplicate matches

Count rows that are identical in every column, and count rows that repeat the same date with the same home and away team.

In [13]:
matches.duplicated().sum()                                   # fully identical rows
matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()   # same match twice

np.int64(0)

### Count missing values

Show how many missing values each column still has.

In [14]:
matches.isna().sum()

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HS          0
AS          0
HST         0
AST         0
HF          0
AF          0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
B365H       0
B365D       0
B365A       0
dtype: int64

In [17]:
matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum()   # expect 0
matches.groupby("Season")["HomeTeam"].nunique()                     # expect 20 per full season
sorted(set(matches["HomeTeam"]))                                    # read the list for spelling variants

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Burnley',
 'Cardiff',
 'Chelsea',
 'Coventry',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Huddersfield',
 'Hull',
 'Ipswich',
 'Leeds',
 'Leicester',
 'Liverpool',
 'Luton',
 'Man City',
 'Man United',
 'Newcastle',
 'Norwich',
 "Nott'm Forest",
 'Sheffield United',
 'Southampton',
 'Stoke',
 'Sunderland',
 'Swansea',
 'Tottenham',
 'Watford',
 'West Brom',
 'West Ham',
 'Wolves']

In [18]:
matches = matches.sort_values(["Date", "HomeTeam"]).reset_index(drop=True)

from pathlib import Path
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
matches.to_csv(out_dir / "matches.csv", index=False)